# Homework 1: Automated Photo Framing & Batch Resizer
For **every** image in a folder: resize to a standard size → grayscale → add a clean border → save to `output/`.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def show(*images, titles=None, size=5):
    """Show one or more images side by side (handles BGR colour and grayscale)."""
    titles = titles or [""] * len(images)
    plt.figure(figsize=(size * len(images), size))
    for k, (im, t) in enumerate(zip(images, titles), 1):
        plt.subplot(1, len(images), k)
        if im.ndim == 2:
            plt.imshow(im, cmap="gray", vmin=0, vmax=255)
        else:
            plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        plt.title(t); plt.axis("off")
    plt.show()

### Get some photos into `photos/`
Run A for samples, or B to upload several of your own.

In [ ]:
# A) sample photos
import os, urllib.request
os.makedirs("photos", exist_ok=True)
for name in ["fruits.jpg", "baboon.jpg", "building.jpg", "butterfly.jpg", "starry_night.jpg"]:
    urllib.request.urlretrieve("https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/" + name, "photos/" + name)
print(os.listdir("photos"))

In [ ]:
# B) upload your own (select several files at once)
import os
from google.colab import files
os.makedirs("photos", exist_ok=True)
for name, data in files.upload().items():
    with open("photos/" + name, "wb") as f:
        f.write(data)
print(os.listdir("photos"))

In [ ]:
import glob, os

SIZE   = (400, 400)   # (width, height)
BORDER = 20
COLOR  = 255

def fit_inside(img, size):
    """Bonus: shrink to fit inside size while keeping the aspect ratio, then pad to exactly size."""
    tw, th = size
    h, w = img.shape[:2]
    s = min(tw / w, th / h)
    nw, nh = max(1, round(w * s)), max(1, round(h * s))
    small = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    top, left = (th - nh) // 2, (tw - nw) // 2
    return cv2.copyMakeBorder(small, top, th - nh - top, left, tw - nw - left,
                              cv2.BORDER_CONSTANT, value=COLOR)

def frame_photo(img, keep_ratio=True):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = fit_inside(gray, SIZE) if keep_ratio else cv2.resize(gray, SIZE, interpolation=cv2.INTER_AREA)
    gray = cv2.copyMakeBorder(gray, 2, 2, 2, 2, cv2.BORDER_CONSTANT, value=40)        # thin dark line
    return cv2.copyMakeBorder(gray, BORDER, BORDER, BORDER, BORDER, cv2.BORDER_CONSTANT, value=COLOR)

os.makedirs("output", exist_ok=True)
paths = sorted(glob.glob("photos/*.jpg") + glob.glob("photos/*.png") + glob.glob("photos/*.jpeg"))
results = []
for path in paths:
    img = cv2.imread(path)
    if img is None:
        print("skipped", path); continue
    framed = frame_photo(img)
    cv2.imwrite("output/" + os.path.basename(path), framed)
    results.append(framed)
    print(f"{os.path.basename(path):20s} {img.shape} -> {framed.shape}")
show(*results[:4], size=3)

In [ ]:
# zip the results and download them (Colab only)
import shutil
from google.colab import files
shutil.make_archive("framed_photos", "zip", "output")
files.download("framed_photos.zip")